# Cálculo del total nacional para junio de 2026

In [0]:
#Leyendo la tabla de características generales del catálogo
cgse=spark.read.format("delta").table("workspace.electiva.cgse")

### -población en edad de trabajar

In [0]:
PET = ft[ft['PET']==1]['FEX_C18'].sum()

###-población que no está en edad de trabajar

### -población económicamente activa 

###-población económicamente inactiva 

### -tasa de ocupación 

### -tasa de desempleo 

In [0]:
import pandas as pd

# Leyendo la tabla de características generales
ft = spark.read.format("delta").table("workspace.electiva.ft")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()

# Primero hay que anexar la P3271 a la tabla de desempleados
desempleados = pd.merge(
    desempleados,
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left'
)

ft = pd.merge(
    ft,
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left'
)

In [0]:
# desempleados_dpto -> cantidad de desempleados por departamento
desempleados_dpto = desempleados[desempleados['DSI'] == 1].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()

In [0]:
desempleados_dpto

In [0]:
# Cantidad de hombres y mujeres económicamente activos por departamento
pea_dpto_genero = ft[ft['FT'] == 1].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()

In [0]:
tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero,'DESEMPLEADOS': desempleados_dpto})

In [0]:
tabla_desempleo

In [0]:
tabla_desempleo['TD'] = (
    tabla_desempleo['DESEMPLEADOS'] / tabla_desempleo['PEA']
) * 100

In [0]:
tabla_desempleo.head(100)

In [0]:
# 4. Diccionarios de mapeo
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO',
    66: 'RISARALDA', 68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA',
    76: 'VALLE DEL CAUCA', 81: 'ARAUCA', 85: 'CASANARE', 86: 'PUTUMAYO',
    88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA', 95: 'GUAVIARE',
    97: 'VAUPES', 99: 'VICHADA'
}

sexo = {1: 'Hombre', 2: 'Mujer'}

In [0]:
tabla_desempleo = tabla_desempleo.reset_index()

tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo)

In [0]:
tabla_desempleo = tabla_desempleo[
    ['Departamento', 'Sexo', 'PEA', 'DESEMPLEADOS', 'TD']
]


In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(
    by=['Sexo', 'TD'],
    ascending=False
)

tabla_desempleo

In [0]:
tabla_desempleo[tabla_desempleo['Sexo'] == 'Mujer']

In [0]:
tabla_desempleo[tabla_desempleo['Sexo'] == 'Hombre']

punto 3

In [0]:
desempleados